# StylePops — Kombin + A/B Üretimi (Colab)

**GPU gerekmez** — varsayılan hızlı mod (CPU, ~5–10 dk). T4 beklenmez.

## Mac (bir kez)
```bash
python scripts/pack_colab_combo_bundle.py
```
→ `outputs/colab_combo_bundle.zip` yükle

## Colab
1. **Çalışma zamanı → Çalışma zamanını fabrika ayarlarına sıfırla**
2. Hücreleri yukarıdan aşağı çalıştır
3. Zip yükle → üretim bitsin → son hücrede indir

GPU varsa üretim hücresinde `USE_FAST = False` yap (FashionCLIP + LoRA).

In [ ]:
import torch

if torch.cuda.is_available():
    print('GPU var (opsiyonel):', torch.cuda.get_device_name(0))
else:
    print('GPU yok — sorun değil, hızlı mod (CPU) kullanılacak')

In [ ]:
# GPU + LoRA kullanacaksan torchao gerekir; CPU hızlı modda atlanabilir
!pip install -q -U pillow joblib lightgbm
!pip install -q -U 'torchao>=0.16.0' peft transformers accelerate torch torchvision 2>/dev/null || true

try:
    import importlib.metadata as _md
    print('torchao:', _md.version('torchao'))
except Exception:
    print('torchao yok — USE_FAST=True ile devam edilir')

In [ ]:
import os

# Yöntem 1 (önerilen): Colab Secrets → HF_TOKEN
# Yöntem 2: Aşağıya yapıştır → HF_TOKEN_MANUAL = 'hf_...'
HF_TOKEN_MANUAL = ''

try:
    from google.colab import userdata
    os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
    os.environ['HUGGING_FACE_HUB_TOKEN'] = os.environ['HF_TOKEN']
    print('HF token: Colab Secrets ✓')
except Exception:
    if HF_TOKEN_MANUAL:
        os.environ['HF_TOKEN'] = HF_TOKEN_MANUAL
        os.environ['HUGGING_FACE_HUB_TOKEN'] = HF_TOKEN_MANUAL
        print('HF token: manuel ✓')
    else:
        print('HF token yok — fashion-clip public, opsiyonel')

In [ ]:
import os, shutil, subprocess
from pathlib import Path

PROJECT = Path('/content/StylePops_Modules')
os.chdir('/content')
if PROJECT.exists():
    shutil.rmtree(PROJECT)
!git clone -q https://github.com/valueisinvalid/StylePops_Modules.git
os.chdir(PROJECT)
!git fetch -q origin main && git reset --hard origin/main
commit = subprocess.check_output(['git', 'rev-parse', '--short', 'HEAD'], text=True).strip()
print('Repo güncel:', PROJECT.resolve(), '| commit:', commit)

In [ ]:
# Mac: outputs/colab_combo_bundle.zip → Dosya seç
from google.colab import files
from pathlib import Path
import os, shutil

if 'PROJECT' not in globals():
    PROJECT = Path('/content/StylePops_Modules')
if not PROJECT.exists():
    raise RuntimeError('Önce clone hücresini çalıştır')

os.chdir(PROJECT)
print('Yükleme klasörü:', PROJECT.resolve())

uploaded = files.upload()
if not uploaded:
    raise FileNotFoundError('Zip seçilmedi. Mac: outputs/colab_combo_bundle.zip')

zip_name = next(iter(uploaded))
ZIP_PATH = PROJECT / zip_name

# Colab bazen /content'e yükler — PROJECT altına taşı
alt = Path('/content') / zip_name
if not ZIP_PATH.exists() and alt.exists():
    shutil.move(str(alt), str(ZIP_PATH))

if not ZIP_PATH.exists():
    raise FileNotFoundError(f'Zip bulunamadı: {ZIP_PATH}')

print('Yüklendi:', ZIP_PATH, f'({ZIP_PATH.stat().st_size // 1024 // 1024} MB)')

In [ ]:
import zipfile, shutil, json
from pathlib import Path

if 'PROJECT' not in globals():
    PROJECT = Path('/content/StylePops_Modules')
if 'ZIP_PATH' not in globals() or not ZIP_PATH.exists():
    raise RuntimeError('Önce zip yükleme hücresini çalıştır')

with zipfile.ZipFile(ZIP_PATH) as zf:
    zf.extractall('/content')

SRC = Path('/content/stylepops_colab')
for rel in ('data/visual/garments_production.json', 'data/visual/inventory_registry.json'):
    shutil.copy2(SRC / rel, PROJECT / rel)
shutil.copytree(SRC / 'data' / 'lookups', PROJECT / 'data' / 'lookups', dirs_exist_ok=True)
for folder in ('data/assets/garments', 'data/assets/fashion_product'):
    src_dir = SRC / folder
    if src_dir.exists():
        shutil.copytree(src_dir, PROJECT / folder, dirs_exist_ok=True)
lora_src = SRC / 'outputs' / 'fashionclip_lora'
if lora_src.exists():
    shutil.copytree(lora_src, PROJECT / 'outputs' / 'fashionclip_lora', dirs_exist_ok=True)
    print('LoRA ✓')

n = json.loads((PROJECT / 'data/visual/garments_production.json').read_text())['count']
print(f'Gardırop hazır: {n} parça')

In [ ]:
import gc, os, runpy, sys, subprocess
from pathlib import Path

# Varsayılan: CPU hızlı mod (GPU/T4 gerekmez)
USE_FAST = True
# GPU + LoRA istiyorsan: USE_FAST = False (torchao>=0.16 gerekir)

PROJECT = Path('/content/StylePops_Modules')
os.chdir(PROJECT)
subprocess.run(['git', 'fetch', 'origin', 'main'], check=False)
subprocess.run(['git', 'reset', '--hard', 'origin/main'], check=False)
print('Git HEAD:', subprocess.check_output(['git', 'rev-parse', '--short', 'HEAD'], text=True).strip())
print('Mod:', 'hızlı (CPU)' if USE_FAST else 'FashionCLIP (GPU önerilir)')

for name in list(sys.modules):
    if name in ('stylepops_core', 'aesthetic_compatibility', 'inventory_loader', 'build_combo_collage', 'garment_eligibility', 'peft'):
        del sys.modules[name]
    elif name.startswith(('stylepops_core', 'peft')):
        del sys.modules[name]

Path('data/assets/combos').mkdir(parents=True, exist_ok=True)
os.environ['PYTHONUNBUFFERED'] = '1'
gc.collect()

args = ['generate_visual_combinations.py', '--ab-pairs', '200', '--n-candidates', '150']
if USE_FAST:
    args.append('--fast')
sys.argv = args
print('Komut:', ' '.join(args))
runpy.run_path('scripts/generate_visual_combinations.py', run_name='__main__')

In [ ]:
import zipfile
from pathlib import Path
from google.colab import files

root = Path('/content/StylePops_Modules')
out = Path('/content/stylepops_results.zip')
with zipfile.ZipFile(out, 'w', zipfile.ZIP_DEFLATED) as zf:
    for name in ('data/visual/combinations_visual.csv', 'data/visual/ab_pairs.csv'):
        zf.write(root / name, Path(name).name)

combos_zip = Path('/content/stylepops_combos.zip')
!cd /content/StylePops_Modules/data/assets/combos && zip -qr {combos_zip} AB*.jpg VC*.jpg

print('İndiriliyor… Mac\'te:')
print('  combinations_visual.csv + ab_pairs.csv → data/visual/')
print('  AB*.jpg VC*.jpg → data/assets/combos/')
files.download(str(out))
files.download(str(combos_zip))